### Image embeddings

We first compute image embeddings and resue throughout evaluation.
Required inputs:

file_name = "interactions_output_file.parquet" <- parquet file containing all interactions

out_dir_name = "images" <- path where images are be stored 

In [ ]:
parquet_file = "interactions_output_file.parquet" 
directory = "images"

Libraries:

In [ ]:
import torch
from PIL import Image
from transformers import AutoModel, AutoProcessor
import open_clip
from pathlib import Path
from tqdm import tqdm
import io
from transformers import AutoProcessor, Siglip2VisionModel
import os
import pandas as pd

device = "cuda" if torch.cuda.is_available() else "cpu"

/Users/georgianamg93gmail.com/interactions/bioclip2-env/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/georgianamg93gmail.com/interactions/bioclip2-env/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Load data:

In [ ]:
def load_data(directory="images", parquet_file = "interactions_output_file.parquet"):

    # read as df
    df = pd.read_parquet(parquet_file,     
                        engine="pyarrow",
                        dtype_backend="pyarrow"
    )

    df["fileName"] = df["image_id"].astype(str) + df["file_ext"].astype(str)
    df["filePath"] = directory + df["fileName"].astype(str)

    # read all filenames present in folder
    files_in_folder = set(os.listdir(directory))

    # keep only rows where df["filename"] exists in folder
    df = df[df["fileName"].isin(files_in_folder)].copy()

    return df

In [8]:
class SigLIP2Wrapper(torch.nn.Module):
    def __init__(self, model_name="google/siglip2-base-patch16-224", device="cpu"):
        super().__init__()
        self.model = AutoModel.from_pretrained(model_name)
        self.processor = AutoProcessor.from_pretrained(model_name)
        self.device = device

    def to(self, device):
        self.device = device
        self.model = self.model.to(device)
        return self

    def eval(self):
        self.model.eval()
        return self

    def encode_image(self, pixel_values):
        return self.model.get_image_features(pixel_values=pixel_values)

    def encode_text(self, input_ids=None, attention_mask=None, **kwargs):
        return self.model.get_text_features(
            input_ids=input_ids,
            attention_mask=attention_mask,
        )


class SigLIP2Preprocess:
    def __init__(self, image_processor):
        self.image_processor = image_processor

    def __call__(self, image):
        out = self.image_processor(images=image, return_tensors="pt")
        return out["pixel_values"].squeeze(0)


class SigLIP2Tokenizer:
    def __init__(self, tokenizer):
        self.tokenizer = tokenizer

    def __call__(self, texts):
        if isinstance(texts, str):
            texts = [texts]
        return self.tokenizer(
            texts,
            return_tensors="pt",
            padding=True,
            truncation=False,
        )

def create_siglip2_model_and_transforms(
    model_name="google/siglip2-base-patch16-224",
    device="cpu",
):
    model = SigLIP2Wrapper(model_name=model_name, device=device)
    preprocess = SigLIP2Preprocess(model.processor.image_processor)
    tokenizer = SigLIP2Tokenizer(model.processor.tokenizer)
    return model, preprocess, preprocess, tokenizer

Prepare a variety of vision-language models (mostly CLIP variants), counts their parameters:

In [ ]:
def get_model_tokenizer(vlm):

    if vlm == "siglip2":
        # Load the SigLip2 as CLIP-like 
        model, preprocess_train, preprocess_val = open_clip.create_model_and_transforms(
            "ViT-L-16-SigLIP2-256",
            pretrained="webli"
        )
        model = model.to(device).eval()
        tokenizer = open_clip.get_tokenizer("ViT-L-16-SigLIP2-256")

    if vlm == 'siglip':
        # Load the SigLip
        model, preprocess_train, preprocess_val = open_clip.create_model_and_transforms(
            "ViT-SO400M-14-SigLIP",
            pretrained="webli",
        )
        model = model.to(device).eval()
        tokenizer = open_clip.get_tokenizer("ViT-SO400M-14-SigLIP")
    
    if vlm == 'bioclip2':
        # Load the BioCLIP 2 model from Hugging Face
        model, preprocess_train, preprocess_val = open_clip.create_model_and_transforms(
            'hf-hub:imageomics/bioclip-2'
        )
        model = model.to(device).eval()
        tokenizer = open_clip.get_tokenizer('hf-hub:imageomics/bioclip-2')

    if vlm == "bioclip":
        # Load the BioCLIP model from Hugging Face
        model, preprocess_train, preprocess_val = open_clip.create_model_and_transforms(
            'hf-hub:imageomics/bioclip'
        )
        model = model.to(device).eval()
        tokenizer = open_clip.get_tokenizer('hf-hub:imageomics/bioclip')
    
    if vlm == "clip":
        # Load the CLIP model from OpenAI
        model, preprocess_train, preprocess_val = open_clip.create_model_and_transforms(
            'ViT-L-14',
            pretrained='openai'
        )
        model = model.to(device).eval()
        tokenizer = open_clip.get_tokenizer('ViT-L-14')

    if vlm == "metaclip":
        model, preprocess_train, preprocess_val = open_clip.create_model_and_transforms(
            "ViT-L-14-quickgelu",
            pretrained="metaclip_fullcc"
        )
        model = model.to(device).eval()
        tokenizer = open_clip.get_tokenizer("ViT-L-14-quickgelu")

    if vlm == "taxabind":
        model, preprocess_train, preprocess_val = open_clip.create_model_and_transforms(
            "hf-hub:MVRL/taxabind-vit-b-16"
        )
        model = model.to(device).eval()
        tokenizer = open_clip.get_tokenizer("hf-hub:MVRL/taxabind-vit-b-16")

    if vlm == "biocap":
        # Load the BioCAP model from Hugging Face
        model, preprocess_train, preprocess_val = open_clip.create_model_and_transforms(
            "hf-hub:imageomics/biocap"
        )
        model = model.to(device).eval()
        tokenizer = open_clip.get_tokenizer("hf-hub:imageomics/biocap")

    if "biotrove" in vlm:
        if "biotrove-bioclip" in vlm:
            model_name = "hf-hub:imageomics/bioclip"
            ckpt_path = "biotrove-clip/biotroveclip-vit-b-16-from-bioclip-epoch-8.pt"
        if "biotrove-metaclip" in vlm:
            model_name = "ViT-L-14"
            ckpt_path = "biotrove-clip/biotroveclip-vit-l-14-from-metaclip-epoch-12.pt"
        if "biotrove-openai" in vlm:
            model_name = "ViT-B-16"
            ckpt_path = "biotrove-clip/biotroveclip-vit-b-16-from-openai-epoch-40.pt"

        model, preprocess_train, preprocess_val = open_clip.create_model_and_transforms(
            model_name,
            pretrained=None,
        )
        tokenizer = open_clip.get_tokenizer(model_name)

        # Load checkpoint
        ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)

        # Extract state dict if nested
        if isinstance(ckpt, dict):
            if "state_dict" in ckpt:
                state_dict = ckpt["state_dict"]
            elif "model_state_dict" in ckpt:
                state_dict = ckpt["model_state_dict"]
            else:
                state_dict = ckpt
        else:
            state_dict = ckpt

        # Remove common prefixes
        cleaned_state_dict = {}
        for k, v in state_dict.items():
            new_k = k
            if new_k.startswith("module."):
                new_k = new_k[len("module."):]
            if new_k.startswith("model."):
                new_k = new_k[len("model."):]
            cleaned_state_dict[new_k] = v

        missing, unexpected = model.load_state_dict(cleaned_state_dict, strict=False)

        print(f"Loaded checkpoint: {ckpt_path}")
        print(f"Missing keys: {len(missing)}")
        print(f"Unexpected keys: {len(unexpected)}")

        model = model.to(device).eval()

    return model, preprocess_val

def get_image_embeddings(model, preprocess_val, batch_size, df):
    emb_list = []
    paths_ok = []
    image_paths = (df["filePath"]).tolist() 
    with torch.no_grad():
        for i in tqdm(range(0, len(image_paths), batch_size)):
            batch = image_paths[i:i+batch_size]

            imgs = []
            ok = []
            for p in batch:
                try:
                    img = preprocess_val(Image.open(p).convert("RGB"))
                    imgs.append(img)
                    ok.append(p)
                except Exception:
                    pass

            if not imgs:
                continue

            imgs = torch.stack(imgs).to(device)
            feats = model.encode_image(imgs)
            feats = feats / feats.norm(dim=-1, keepdim=True)  # normalize

            emb_list.append(feats.cpu())
            paths_ok.extend(ok)
            
        return emb_list, paths_ok
    
def save_embeddings(df, emb_list, paths_ok, folder, vlm, name):

    emb = torch.cat(emb_list, dim=0).numpy()  # (N, D)
    print("Embeddings:", emb.shape)

    subset = df[df["filePath"].isin(paths_ok)].reset_index(drop=True)

    torch.save({"emb": emb, "df": subset}, f"image_embeddings_{vlm}.pt")    

Load embeddings:

In [ ]:
df = load_data(directory=directory, parquet_file=parquet_file)

vlms = ["bioclip", "bioclip2", "siglip", "siglip2", "clip", "metaclip", "biocap", "taxabind", "biotrove-bioclip", "biotrove-openai", "biotrove-metaclip"]

for vlm in vlms:

    model, preprocess_val = get_model_tokenizer(vlm)
    
    print("Using device:", device)

    # Get embeddings
    model = model.to(device).eval()

    batch_size = 32

    emb_list,paths_ok = get_image_embeddings(model, preprocess_val, batch_size, df)

    save_embeddings(df, emb_list, paths_ok, vlm)
